# Impact of Spatial Criterion
## 1. Setup

In [1]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.getcwd())
import config

A_MIN = config.A_MIN

m_per_deg = config.SCREEN_DIST_M * np.tan(np.radians(1))
PPD_X = config.SCREEN_RES[0]/config.SCREEN_SIZE_M[0] * m_per_deg
PPD_Y = config.SCREEN_RES[1]/config.SCREEN_SIZE_M[1] * m_per_deg
print(f"A_MIN={A_MIN} deg, PPD_X={PPD_X:.2f} PPD_Y={PPD_Y:.2f} px/deg")

A_MIN=1.0 deg, PPD_X=44.26 PPD_Y=44.76 px/deg


## 2. Load pre-merge data (sub-005)

In [31]:
DATA_PATH = r"C:\Users\chris\Documents\ArbeitUni\VIS_S-CCS\FreeViewing\BIDS\derivatives\in\sub-005\ses-001\misc\sub-005_ses-001_task-freeviewing_et_events.tsv"
df = pd.read_csv(DATA_PATH, sep="\t")
print("length:", len(df))
print(df.trial_type.value_counts().to_dict())

length: 32948
{'fixation': 15179, 'saccade': 15159, 'blink': 2610}


## 3. Merge function WITH spatial criterion

In [36]:
def interfix_dist_deg(a, b):
    """Distance between two fixation centroids, in degrees (per-axis px->deg)."""
    dx = (b["fix_avg_x"] - a["fix_avg_x"]) / PPD_X
    dy = (b["fix_avg_y"] - a["fix_avg_y"]) / PPD_Y
    return np.hypot(dx, dy)

def merge_fixation_candidates_w_spatial(events, a_min=A_MIN, merge_threshold=None):
    events = events.copy()
    t_min_sac = (2.2 * a_min + 27) / 1000.0
    merge_threshold = t_min_sac if merge_threshold is None else merge_threshold/1000.0
    events = events.sort_values(["eye", "onset"]).reset_index(drop=True)
    events = events[~(
        (events["trial_type"] == "saccade")
        & (events["sacc_visual_angle"] < a_min)
        & (events["duration"] < t_min_sac)
    )].reset_index(drop=True)

    fixations_meeting_criterion = []
    rows, i = [], 0
    while i < len(events):
        cur = events.iloc[i].copy()
        if (i < len(events) - 1
                and events.iloc[i]["trial_type"] == "fixation"
                and events.iloc[i+1]["trial_type"] == "fixation"
                and events.iloc[i]["eye"] == events.iloc[i+1]["eye"]
                and events.iloc[i+1]["onset"] - events.iloc[i]["end_time"] < merge_threshold
                and interfix_dist_deg(events.iloc[i], events.iloc[i+1]) < a_min # CRITERION
                ):
            j = i + 1
            dur_sum = cur["duration"]
            fixations_meeting_criterion.append(i)
            while (j < len(events)
                   and events.iloc[j]["trial_type"] == "fixation"
                   and events.iloc[j]["eye"] == cur["eye"]
                   and events.iloc[j]["onset"] - events.iloc[j-1]["end_time"] < merge_threshold
                   and interfix_dist_deg(events.iloc[i], events.iloc[i+1]) < a_min # CRITERION
                   ):
                next = events.iloc[j]
                dur_sum += next["duration"]
                for c in ["fix_avg_x", "fix_avg_y", "fix_avg_pupil_size"]:
                    cur[c] = (cur[c]*(dur_sum-next["duration"]) + next[c]*next["duration"]) / dur_sum
                j += 1
                fixations_meeting_criterion.append(j)
            cur["end_time"] = next["end_time"]
            cur["duration"] = next["end_time"] - cur["onset"]
            rows.append(cur)
            i = j
        else:
            rows.append(cur)
            i += 1
    return pd.DataFrame(rows).sort_values("onset").reset_index(drop=True), fixations_meeting_criterion

## 4. Merge function WITHOUT spatial criterion

In [37]:
def merge_fixation_candidates_wo_spatial(events, a_min=A_MIN, merge_threshold=None):
    events = events.copy()
    t_min_sac = (2.2 * a_min + 27) / 1000.0
    merge_threshold = t_min_sac if merge_threshold is None else merge_threshold/1000.0
    events = events.sort_values(["eye", "onset"]).reset_index(drop=True)
    events = events[~(
        (events["trial_type"] == "saccade")
        & (events["sacc_visual_angle"] < a_min)
        & (events["duration"] < t_min_sac)
    )].reset_index(drop=True)

    fixations_meeting_criterion = []
    rows, i = [], 0
    while i < len(events):
        cur = events.iloc[i].copy()
        if (i < len(events) - 1
                and events.iloc[i]["trial_type"] == "fixation"
                and events.iloc[i+1]["trial_type"] == "fixation"
                and events.iloc[i]["eye"] == events.iloc[i+1]["eye"]
                and events.iloc[i+1]["onset"] - events.iloc[i]["end_time"] < merge_threshold
                # and interfix_dist_deg(events.iloc[i], events.iloc[i+1]) < a_min
                ):
            j = i + 1
            dur_sum = cur["duration"]
            fixations_meeting_criterion.append(i)
            while (j < len(events)
                   and events.iloc[j]["trial_type"] == "fixation"
                   and events.iloc[j]["eye"] == cur["eye"]
                   and events.iloc[j]["onset"] - events.iloc[j-1]["end_time"] < merge_threshold
                   # and interfix_dist_deg(events.iloc[i], events.iloc[i+1]) < a_min
                   ):
                next = events.iloc[j]
                dur_sum += next["duration"]
                for c in ["fix_avg_x", "fix_avg_y", "fix_avg_pupil_size"]:
                    cur[c] = (cur[c]*(dur_sum-next["duration"]) + next[c]*next["duration"]) / dur_sum
                j += 1
                fixations_meeting_criterion.append(j)
            cur["end_time"] = next["end_time"]
            cur["duration"] = next["end_time"] - cur["onset"]
            rows.append(cur)
            i = j
        else:
            rows.append(cur)
            i += 1
    return pd.DataFrame(rows).sort_values("onset").reset_index(drop=True), fixations_meeting_criterion

In [38]:
m_with, criterion_met_w = merge_fixation_candidates_w_spatial(df, A_MIN, merge_threshold=100)
m_without, criterion_met_wo = merge_fixation_candidates_wo_spatial(df, A_MIN, merge_threshold=100)

In [39]:
print(f"Number of fixations: ",
      f"\n\t with spatial criterion: {m_with["trial_type"].value_counts()["fixation"]}, merges: {len(criterion_met_w)}",
      f"\n\t without spatial criterion: {m_without["trial_type"].value_counts()["fixation"]}, merges: {len(criterion_met_wo)}",
      f"\nDifference: {abs(m_with["trial_type"].value_counts()["fixation"] - m_without["trial_type"].value_counts()["fixation"])}"
)

Number of fixations:  
	 with spatial criterion: 12116, merges: 5409 
	 without spatial criterion: 11685, merges: 6147 
Difference: 431


In [7]:
m_with

,onset,duration,end_time,trial_type,eye,fix_avg_x,fix_avg_y,fix_avg_pupil_size,sacc_start_x,sacc_start_y,sacc_end_x,sacc_end_y,sacc_visual_angle,peak_velocity
0,190.768637,0.407,191.174637,fixation,L,968.500000,532.600000,906.000000,NaN,NaN,NaN,NaN,NaN,NaN
1,190.768637,0.407,191.174637,fixation,R,964.400000,515.400000,787.000000,NaN,NaN,NaN,NaN,NaN,NaN
2,191.175637,0.024,191.198637,saccade,R,NaN,NaN,NaN,962.1,516.8,873.2,499.8,2.04,112.0
3,191.175637,0.032,191.206637,saccade,L,NaN,NaN,NaN,965.1,532.4,845.5,514.4,2.73,118.0
4,191.199637,0.311,191.510637,fixation,R,849.333333,503.366667,827.666667,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26388,3560.570910,0.083,3560.652910,fixation,L,485.000000,1110.600000,1339.000000,NaN,NaN,NaN,NaN,NaN,NaN
26389,3560.653910,0.064,3560.716910,saccade,L,NaN,NaN,NaN,485.2,1117.9,930.5,560.0,15.40,422.0
26390,3560.654910,0.065,3560.718910,saccade,R,NaN,NaN,NaN,477.2,998.8,974.8,508.9,15.10,429.0
26391,3560.717910,0.134,3560.850910,fixation,L,924.900000,559.000000,1343.000000,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
m_without

,onset,duration,end_time,trial_type,eye,fix_avg_x,fix_avg_y,fix_avg_pupil_size,sacc_start_x,sacc_start_y,sacc_end_x,sacc_end_y,sacc_visual_angle,peak_velocity
0,190.768637,0.407,191.174637,fixation,L,968.500000,532.600000,906.000000,NaN,NaN,NaN,NaN,NaN,NaN
1,190.768637,0.407,191.174637,fixation,R,964.400000,515.400000,787.000000,NaN,NaN,NaN,NaN,NaN,NaN
2,191.175637,0.024,191.198637,saccade,R,NaN,NaN,NaN,962.1,516.8,873.2,499.8,2.04,112.0
3,191.175637,0.032,191.206637,saccade,L,NaN,NaN,NaN,965.1,532.4,845.5,514.4,2.73,118.0
4,191.199637,0.311,191.510637,fixation,R,849.333333,503.366667,827.666667,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25957,3560.570910,0.083,3560.652910,fixation,L,485.000000,1110.600000,1339.000000,NaN,NaN,NaN,NaN,NaN,NaN
25958,3560.653910,0.064,3560.716910,saccade,L,NaN,NaN,NaN,485.2,1117.9,930.5,560.0,15.40,422.0
25959,3560.654910,0.065,3560.718910,saccade,R,NaN,NaN,NaN,477.2,998.8,974.8,508.9,15.10,429.0
25960,3560.717910,0.134,3560.850910,fixation,L,924.900000,559.000000,1343.000000,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Possibly wrongly-merged pairs (computed post-drop)
= pairs that are close in time (< merge_threshold) but >= `a_min` distance in space

In [9]:
m_wo_fix = m_without[m_without["trial_type"] == "fixation"]
m_w_fix  = m_with[m_with["trial_type"] == "fixation"]

diff_df = m_wo_fix.merge(m_w_fix, how="left", indicator=True).query('_merge == "left_only"').drop(columns=["_merge"])
diff_df

,onset,duration,end_time,trial_type,eye,fix_avg_x,fix_avg_y,fix_avg_pupil_size,sacc_start_x,sacc_start_y,sacc_end_x,sacc_end_y,sacc_visual_angle,peak_velocity
107,209.544633,0.279,209.823633,fixation,L,835.141445,477.231939,839.574144,NaN,NaN,NaN,NaN,NaN,NaN
140,216.281631,0.324,216.605631,fixation,L,1351.005145,255.999678,1101.717042,NaN,NaN,NaN,NaN,NaN,NaN
197,335.845606,1.768,337.613605,fixation,R,954.700399,511.230177,886.762693,NaN,NaN,NaN,NaN,NaN,NaN
198,335.846606,1.766,337.612605,fixation,L,927.663402,527.376060,1000.154639,NaN,NaN,NaN,NaN,NaN,NaN
237,344.845604,1.246,346.091603,fixation,R,958.882482,493.317032,951.759124,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11542,3533.426916,0.869,3534.295915,fixation,R,1010.147468,670.393757,475.891637,NaN,NaN,NaN,NaN,NaN,NaN
11543,3534.316915,0.617,3534.933915,fixation,L,1051.290216,600.126036,524.475954,NaN,NaN,NaN,NaN,NaN,NaN
11544,3534.319915,0.615,3534.934915,fixation,R,1009.075630,618.032773,463.260504,NaN,NaN,NaN,NaN,NaN,NaN
11639,3554.316911,0.430,3554.746911,fixation,L,873.405742,563.711483,1190.167464,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
diff_outer = m_wo_fix.merge(m_w_fix, how="outer", indicator=True)

only_in_without = (diff_outer['_merge'] == 'left_only').sum()
only_in_with    = (diff_outer['_merge'] == 'right_only').sum()
both            = (diff_outer['_merge'] == 'both').sum()

print(f"Exklusiv in 'without': {only_in_without}") # Das sind deine 409!
print(f"Exklusiv in 'with':    {only_in_with}")
print(f"In beiden vorhanden:   {both}")
print("---")
print(f"Rechnerische Netto-Differenz: {only_in_with - only_in_without}")

diff_outer

Exklusiv in 'without': 409
Exklusiv in 'with':    840
In beiden vorhanden:   11276
---
Rechnerische Netto-Differenz: 431


,onset,duration,end_time,trial_type,eye,fix_avg_x,fix_avg_y,fix_avg_pupil_size,sacc_start_x,sacc_start_y,sacc_end_x,sacc_end_y,sacc_visual_angle,peak_velocity,_merge
0,190.768637,0.407,191.174637,fixation,L,968.500000,532.600000,906.000000,NaN,NaN,NaN,NaN,NaN,NaN,both
1,190.768637,0.407,191.174637,fixation,R,964.400000,515.400000,787.000000,NaN,NaN,NaN,NaN,NaN,NaN,both
2,191.199637,0.311,191.510637,fixation,R,849.333333,503.366667,827.666667,NaN,NaN,NaN,NaN,NaN,NaN,both
3,191.207637,0.304,191.511637,fixation,L,846.532653,519.537415,946.591837,NaN,NaN,NaN,NaN,NaN,NaN,both
4,191.534637,0.368,191.901637,fixation,R,929.100000,531.100000,717.000000,NaN,NaN,NaN,NaN,NaN,NaN,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12520,3560.293910,0.107,3560.399910,fixation,L,1000.600000,523.600000,1209.000000,NaN,NaN,NaN,NaN,NaN,NaN,both
12521,3560.542910,0.112,3560.653910,fixation,R,473.100000,987.100000,1312.000000,NaN,NaN,NaN,NaN,NaN,NaN,both
12522,3560.570910,0.083,3560.652910,fixation,L,485.000000,1110.600000,1339.000000,NaN,NaN,NaN,NaN,NaN,NaN,both
12523,3560.717910,0.134,3560.850910,fixation,L,924.900000,559.000000,1343.000000,NaN,NaN,NaN,NaN,NaN,NaN,both


## Visualise the worst cases: two distinct fixations a time-only rule would fuse

In [11]:
ex = critical.sort_values("dist_deg", ascending=False).head(3)
fig, axes = plt.subplots(len(ex),1, figsize=(11,3*len(ex))); axes=np.atleast_1d(axes)
for ax,(_,r) in zip(axes, ex.iterrows()):
    i=int(r.idx); a=ev.iloc[i]; b=ev.iloc[i+1]
    ax.hlines(a.fix_avg_x, a.onset, a.end_time, colors="#0072B2", lw=6, label="fixation A")
    ax.hlines(b.fix_avg_x, b.onset, b.end_time, colors="#E69F00", lw=6, label="fixation B")
    mx=(a.fix_avg_x*a.duration + b.fix_avg_x*b.duration)/(a.duration+b.duration)
    ax.hlines(mx, a.onset, b.end_time, colors="crimson", lw=2, ls="--", label="merged (no-spatial)")
    ax.set_title(f"eye {r.eye}: gap={r.gap_ms:.0f} ms (<{MERGE_THRESHOLD*1000:.0f}) but {r.dist_deg:.2f} deg apart (>= {A_MIN})")
    ax.set_xlabel("time (s)"); ax.set_ylabel("horiz pos (px)"); ax.legend(fontsize=8)
fig.tight_layout(); plt.show()

critical.sort_values("dist_deg", ascending=False)

NameError: name 'critical' is not defined

## Reference — your CURRENT code, unmodified
Runs the live `merge_fixation_candidates` from `preprocessing.py` on the same data. Its spatial line
measures each fixation's distance from the origin in pixels (`hypot(x, y) < a_min`), which is never
true, so its outer merge condition never fires — it merges nothing.

In [ ]:
import importlib, preprocessing; importlib.reload(preprocessing)
cur = preprocessing.merge_fixation_candidates(df.copy(), a_min=A_MIN)
print("input fixations (pre-merge)            :", (df.trial_type=='fixation').sum())
print("current code (buggy spatial)           :", (cur.trial_type=='fixation').sum(), " <- no merges")
print("corrected WITH spatial                 :", (m_with.trial_type=='fixation').sum())
print("corrected WITHOUT spatial              :", (m_without.trial_type=='fixation').sum())

input fixations (pre-merge)            : 15179
current code (buggy spatial)           : 15179  <- no merges
corrected WITH spatial                 : 12217
corrected WITHOUT spatial              : 11685


## Summary
- On raw `sub-005`, the drop step removes ~3,500 small-short saccades, creating ~3,510 adjacent
  fixation pairs; the merger is active.
- **Corrected WITH vs WITHOUT spatial** differ by the time-close / space-far pairs (~530), which span
  1.0–3.3 deg apart — real spatial jumps the time-only rule would wrongly fuse. So the spatial
  criterion is doing meaningful work.
- Your **current unmodified code merges nothing**, because its spatial line tests distance-from-origin
  in pixels against `a_min` in degrees (always false). That's the bug to fix.

Change `SUBJECT` to test another subject.